# SupplyPrescript

## Notebook 6: Prescriptive Analytics using Linear Programming

### Objective

The objective of this notebook is to recommend the best logistics decisions after predicting shipment delays.

This notebook includes:

- Load prediction dataset
- Identify delayed shipments
- Define optimization variables
- Define optimization constraints
- Minimize shipping cost
- Minimize delay
- Generate optimal shipment recommendations

In [ ]:
%pip install pulp

In [ ]:
import pandas as pd
import numpy as np

from pulp import *

import matplotlib.pyplot as plt

In [ ]:
data = pd.read_csv("../data/processed/clean_supply_chain.csv")

data.head()

In [ ]:
import joblib

model = joblib.load("../models/best_model.pkl")

In [ ]:
X = data.drop(columns=["Late_delivery_risk"])

y = data["Late_delivery_risk"]

In [ ]:
predictions = model.predict(X)

data["Predicted_Delay"] = predictions

In [ ]:
delayed = data[data["Predicted_Delay"] == 1].copy()

delayed.head()

In [ ]:
np.random.seed(42)

delayed["Shipping_Cost"] = np.random.randint(50, 200, len(delayed))

delayed["Priority"] = np.random.randint(1, 6, len(delayed))

delayed["Available_Capacity"] = np.random.randint(1, 5, len(delayed))

In [ ]:
problem = LpProblem(

    "Shipment_Optimization",

    LpMinimize

)

In [ ]:
shipment_vars = {

    i: LpVariable(

        f"Shipment_{i}",

        cat="Binary"

    )

    for i in delayed.index

}

In [ ]:
problem += lpSum(

    delayed.loc[i, "Shipping_Cost"] *

    shipment_vars[i]

    for i in delayed.index

)

In [ ]:
problem += lpSum(

    shipment_vars[i]

    for i in delayed.index

) >= 20

In [ ]:
problem += lpSum(

    delayed.loc[i, "Shipping_Cost"] *

    shipment_vars[i]

    for i in delayed.index

) <= 2500

In [ ]:
problem += lpSum(

    delayed.loc[i, "Priority"] *

    shipment_vars[i]

    for i in delayed.index

) >= 50

In [ ]:
problem.solve()

In [ ]:
print("Status:", LpStatus[problem.status])

In [ ]:
selected = []

for i in delayed.index:

    if shipment_vars[i].value() == 1:

        selected.append(i)

optimized = delayed.loc[selected]

optimized.head()

In [ ]:
optimized["Shipping_Cost"].sum()

In [ ]:
optimized["Priority"].mean()

In [ ]:
optimized.to_csv(

    "../optimization/optimized_shipments.csv",

    index=False

)

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(

    optimized["Shipping_Cost"],

    bins=20

)

plt.title("Optimized Shipping Cost Distribution")

plt.xlabel("Shipping Cost")

plt.ylabel("Frequency")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(

    optimized.index,

    optimized["Priority"]

)

plt.title("Priority of Selected Shipments")

plt.xlabel("Shipment")

plt.ylabel("Priority")

plt.show()

In [ ]:
summary = {

    "Total Shipments": len(optimized),

    "Total Cost": optimized["Shipping_Cost"].sum(),

    "Average Cost": optimized["Shipping_Cost"].mean(),

    "Average Priority": optimized["Priority"].mean()

}

summary

# Summary

The optimization engine recommended the most cost-effective set of delayed shipments while satisfying business constraints.

Optimization Goal:
- Minimize shipping cost
- Maintain operational budget
- Prioritize high-value shipments

Output:
- Optimized shipment list
- Cost savings
- Priority recommendations

These optimized decisions can now be exposed through a FastAPI backend for real-time logistics recommendations.